<a href="https://colab.research.google.com/github/Lydia-fadele/AfroDiabDB/blob/main/notebooks/AfroDiabDB_v1.2_Chapter3_Workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:

# ==============================================================================
# AFRODIABDB v1.3: CHAPTER 3 WORKFLOW
# Step 1: Software Environment & Package Setup
# ==============================================================================

!pip install -q rdkit openpyxl matplotlib seaborn scikit-learn scipy pandas numpy graphviz

import os
import glob
import shutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import graphviz

# RDKit Modules
import rdkit
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, QED, AllChem

# Scikit-Learn Modules
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Set Global Styling for Publication Quality (300 DPI)
sns.set_theme(style="ticks", palette="muted")
plt.rcParams['font.sans-serif'] = "DejaVu Sans"
plt.rcParams['font.size'] = 11
plt.rcParams['figure.dpi'] = 300

# Global dictionary to hold all 16 generated summary tables
tables_dict = {}

print(f"✅ Environment initialized! RDKit Version: {rdkit.__version__}")

✅ Environment initialized! RDKit Version: 2026.03.5


In [4]:

# ==============================================================================
# Step 2: Interactive Dataset Upload
# ==============================================================================

from google.colab import files

print("📁 Please select and upload your dataset file below:")
uploaded = files.upload()

file_name = list(uploaded.keys())[0]

if file_name.endswith('.csv'):
    df = pd.read_csv(file_name)
else:
    df = pd.read_excel(file_name)

print(f"\n✅ Dataset '{file_name}' successfully loaded!")
print(f"📊 Total Records: {df.shape[0]} | Total Columns: {df.shape[1]}")

📁 Please select and upload your dataset file below:


Saving AfroDiabDB_v1.2_final.xlsx to AfroDiabDB_v1.2_final (2).xlsx

✅ Dataset 'AfroDiabDB_v1.2_final (2).xlsx' successfully loaded!
📊 Total Records: 248 | Total Columns: 37


In [5]:

# ==============================================================================
# Step 3: Database Overview & Architecture
# ==============================================================================

# --- Table 3.1: Overview of AfroDiabDB v1.3 ---
table_3_1 = pd.DataFrame({
    "Metric Description": [
        "Total Occurrence Records",
        "Unique Compounds",
        "Unique SMILES",
        "Plant Species",
        "Plant Families",
        "Validated PubChem CIDs",
        "InChIKeys Generated"
    ],
    "Value": [
        len(df),
        df['Compound_Name'].nunique() if 'Compound_Name' in df.columns else len(df),
        df['Canonical_SMILES'].nunique() if 'Canonical_SMILES' in df.columns else len(df),
        df['Plant_Name'].nunique() if 'Plant_Name' in df.columns else 0,
        df['Plant_Family'].nunique() if 'Plant_Family' in df.columns else 0,
        df['PubChem_CID'].dropna().nunique() if 'PubChem_CID' in df.columns else len(df),
        df['Canonical_SMILES'].nunique() if 'Canonical_SMILES' in df.columns else len(df)
    ]
})

tables_dict['Table_3_1_Database_Overview'] = table_3_1
print("=== TABLE 3.1: DATABASE OVERVIEW ===")
print(table_3_1.to_string(index=False))

# --- Figure 3.1: AfroDiabDB Construction Workflow ---
dot = graphviz.Digraph('workflow', format='png')
dot.attr(rankdir='LR', size='10,3', dpi='300')
dot.attr('node', shape='rectangle', style='filled', fillcolor='#EBF5FB', color='#2E86C1', fontname='Helvetica', fontsize='10')

steps = [
    "Literature Mining", "Data Extraction", "Structure Validation",
    "Descriptor Calculation", "Drug-Likeness Analysis", "PCA Analysis", "Database Deployment"
]

for i, step in enumerate(steps):
    dot.node(f"S{i}", step)
    if i > 0:
        dot.edge(f"S{i-1}", f"S{i}")

dot.render('Figure_3_1_Workflow', cleanup=True)
print("✅ Figure 3.1 generated: Figure_3_1_Workflow.png")

=== TABLE 3.1: DATABASE OVERVIEW ===
      Metric Description  Value
Total Occurrence Records    248
        Unique Compounds    217
           Unique SMILES    202
           Plant Species     51
          Plant Families     28
  Validated PubChem CIDs    190
     InChIKeys Generated    202
✅ Figure 3.1 generated: Figure_3_1_Workflow.png


In [6]:

# ==============================================================================
# Step 4: Taxonomic & Botanical Distribution
# ==============================================================================

# --- Table 3.2: Top Medicinal Plant Species ---
top_species = df['Plant_Name'].value_counts().head(10).reset_index()
top_species.columns = ['Plant Species', 'Number of Compounds']
top_species['Percentage (%)'] = ((top_species['Number of Compounds'] / len(df)) * 100).round(2)

tables_dict['Table_3_2_Top_Species'] = top_species
print("=== TABLE 3.2: TOP MEDICINAL PLANT SPECIES ===")
print(top_species.to_string(index=False))

# --- Table 3.3 & Figure 3.2: Botanical Family Distribution ---
family_dist = df['Plant_Family'].value_counts().reset_index()
family_dist.columns = ['Family', 'Records']
family_dist['Percentage (%)'] = ((family_dist['Records'] / len(df)) * 100).round(2)

tables_dict['Table_3_3_Family_Distribution'] = family_dist

plt.figure(figsize=(10, 5))
top_10_fam = family_dist.head(10)
sns.barplot(data=top_10_fam, y='Family', x='Records', palette='Blues_r')
plt.title('Figure 3.2: Distribution of Top 10 Botanical Families', fontweight='bold', pad=12)
plt.xlabel('Number of Records')
plt.ylabel('Plant Family')
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

for p in plt.gca().patches:
    width = p.get_width()
    plt.gca().annotate(f'{int(width)}', (width + 0.5, p.get_y() + p.get_height() / 2.), va='center')

plt.tight_layout()
plt.savefig('Figure_3_2_Plant_Families.png', dpi=300)
plt.close()

# --- Figure 3.3: Plant Part Utilization Frequency ---
parts_dist = df['Plant_Part'].value_counts().head(8).reset_index()
parts_dist.columns = ['Plant Part', 'Count']

plt.figure(figsize=(9, 4.5))
sns.barplot(data=parts_dist, y='Plant Part', x='Count', palette='Greens_r')
plt.title('Figure 3.3: Plant Part Utilization Frequency', fontweight='bold', pad=12)
plt.xlabel('Frequency')
plt.ylabel('Plant Part')
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

for p in plt.gca().patches:
    width = p.get_width()
    plt.gca().annotate(f'{int(width)}', (width + 0.3, p.get_y() + p.get_height() / 2.), va='center')

plt.tight_layout()
plt.savefig('Figure_3_3_Plant_Parts.png', dpi=300)
plt.close()

print("✅ Table 3.2, Table 3.3, Figure 3.2, and Figure 3.3 generated.")

=== TABLE 3.2: TOP MEDICINAL PLANT SPECIES ===
           Plant Species  Number of Compounds  Percentage (%)
        Carica papaya L.                   43           17.34
        Moringa oleifera                   35           14.11
     Momordica charantia                   20            8.06
          Carissa edulis                   19            7.66
      Dracaena steudneri                   13            5.24
         Acacia nilotica                   11            4.44
   Parquetina nigrescens                   10            4.03
             Allium cepa                    9            3.63
Momordica balsamina Linn                    8            3.23
     Dovyalis abyssinica                    7            2.82


/tmp/ipykernel_20726/1229905301.py:23: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=top_10_fam, y='Family', x='Records', palette='Blues_r')
/tmp/ipykernel_20726/1229905301.py:43: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=parts_dist, y='Plant Part', x='Count', palette='Greens_r')


✅ Table 3.2, Table 3.3, Figure 3.2, and Figure 3.3 generated.


In [7]:

# ==============================================================================
# Step 5: Chemical Class Diversity
# ==============================================================================

class_col = [c for c in df.columns if 'compound_class' in c.lower() or c.lower() == 'class'][0]
class_counts = df[class_col].value_counts().reset_index()
class_counts.columns = ['Compound Class', 'Frequency']
class_counts['Percentage (%)'] = ((class_counts['Frequency'] / len(df)) * 100).round(2)

tables_dict['Table_3_4_Chemical_Classes'] = class_counts

# --- Figure 3.4: Pie Chart of Phytochemical Classes ---
top_classes = class_counts.head(6).copy()
other_count = class_counts['Frequency'][6:].sum()
top_classes.loc[len(top_classes)] = ['Others', other_count, (other_count/len(df)*100).round(2)]

plt.figure(figsize=(7, 7))
plt.pie(top_classes['Frequency'], labels=top_classes['Compound Class'], autopct='%1.1f%%',
        startangle=140, colors=sns.color_palette('pastel'))
plt.title('Figure 3.4: Distribution of Phytochemical Classes', fontweight='bold')
plt.tight_layout()
plt.savefig('Figure_3_4_Chemical_Classes.png', dpi=300)
plt.close()

print("✅ Table 3.4 and Figure 3.4 generated.")

✅ Table 3.4 and Figure 3.4 generated.


In [8]:
# ==============================================================================
# Step 6: Molecular Descriptors & Statistical Profiling
# ==============================================================================

descriptor_cols = [
    'Molecular_Weight', 'LogP', 'TPSA', 'HBA', 'HBD', 'Rotatable_Bonds',
    'Molar_Refractivity', 'Fraction_CSP3', 'Ring_Count', 'Aromatic_Ring_Count', 'Heavy_Atom_Count'
]

resolved_desc = [c for c in descriptor_cols if c in df.columns]

# --- Table 3.5: Descriptor Descriptive Statistics ---
desc_stats = df[resolved_desc].describe().T[['mean', 'std', '50%', 'min', 'max']]
desc_stats.columns = ['Mean', 'SD', 'Median', 'Min', 'Max']
desc_stats = desc_stats.round(2).reset_index()
desc_stats.rename(columns={'index': 'Descriptor'}, inplace=True)

tables_dict['Table_3_5_Descriptor_Statistics'] = desc_stats
print("=== TABLE 3.5: DESCRIPTOR STATISTICS ===")
print(desc_stats.to_string(index=False))

# --- Standalone Histograms: Figures 3.5, 3.6, 3.7 ---
# Figure 3.5: MW Distribution
plt.figure(figsize=(6, 4))
sns.histplot(df['Molecular_Weight'], kde=True, color='#1f77b4', bins=20)
plt.title('Figure 3.5: Molecular Weight (MW) Distribution', fontweight='bold')
plt.xlabel('Molecular Weight (g/mol)')
plt.ylabel('Frequency')
plt.savefig('Figure_3_5_MW_Distribution.png', dpi=300)
plt.close()

# Figure 3.6: LogP Distribution
plt.figure(figsize=(6, 4))
sns.histplot(df['LogP'], kde=True, color='#2ca02c', bins=20)
plt.title('Figure 3.6: LogP Distribution', fontweight='bold')
plt.xlabel('LogP')
plt.ylabel('Frequency')
plt.savefig('Figure_3_6_LogP_Distribution.png', dpi=300)
plt.close()

# Figure 3.7: TPSA Distribution
plt.figure(figsize=(6, 4))
sns.histplot(df['TPSA'], kde=True, color='#d62728', bins=20)
plt.title('Figure 3.7: TPSA Distribution', fontweight='bold')
plt.xlabel('TPSA (Å²)')
plt.ylabel('Frequency')
plt.savefig('Figure_3_7_TPSA_Distribution.png', dpi=300)
plt.close()

# --- Figure 3.8: Correlation Heatmap ---
plt.figure(figsize=(10, 8))
corr_matrix = df[resolved_desc].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5)
plt.title('Figure 3.8: Correlation Heatmap of Molecular Descriptors', fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('Figure_3_8_Correlation_Heatmap.png', dpi=300)
plt.close()

print("✅ Table 3.5 and Figures 3.5–3.8 generated.")

=== TABLE 3.5: DESCRIPTOR STATISTICS ===
         Descriptor   Mean     SD  Median   Min     Max
   Molecular_Weight 339.74 155.01  299.28 76.10 1457.57
               LogP   1.90   2.98    1.55 -6.55   15.83
               TPSA 110.57  76.61   90.90  0.00  494.88
                HBA   6.15   4.66    5.00  0.00   31.00
                HBD   3.81   2.92    3.00  0.00   16.00
    Rotatable_Bonds   3.92   4.22    2.00  0.00   31.00
 Molar_Refractivity  87.76  38.27   78.34 18.77  343.82
      Fraction_CSP3   0.37   0.32    0.29  0.00    1.00
         Ring_Count   2.77   1.71    3.00  0.00   11.00
Aromatic_Ring_Count   1.56   1.19    2.00  0.00    4.00
   Heavy_Atom_Count  24.36  10.97   22.00  5.00  102.00
✅ Table 3.5 and Figures 3.5–3.8 generated.


In [9]:
# ==============================================================================
# Step 7: Drug-Likeness Rules Evaluation
# ==============================================================================

def calc_filter_summary(col_name):
    if col_name in df.columns:
        counts = df[col_name].value_counts()
        total = len(df)
        pass_cnt = counts.get('Pass', counts.get('Accepted', 0))
        fail_cnt = total - pass_cnt
        return pd.DataFrame({
            'Status': ['Pass', 'Fail'],
            'Number': [pass_cnt, fail_cnt],
            'Percentage (%)': [round((pass_cnt/total)*100, 2), round((fail_cnt/total)*100, 2)]
        })
    return pd.DataFrame({'Status': ['Pass', 'Fail'], 'Number': [0, 0], 'Percentage (%)': [0.0, 0.0]})

# --- Tables 3.6, 3.7, 3.8 ---
table_3_6 = calc_filter_summary('Lipinski_Status')
table_3_7 = calc_filter_summary('Veber_Status')
table_3_8 = calc_filter_summary('Ghose_Status')

tables_dict['Table_3_6_Lipinski'] = table_3_6
tables_dict['Table_3_7_Veber'] = table_3_7
tables_dict['Table_3_8_Ghose'] = table_3_8

# --- Table 3.9: Lipinski Violation Distribution ---
if 'Lipinski_Violations' in df.columns:
    table_3_9 = df['Lipinski_Violations'].value_counts().sort_index().reset_index()
    table_3_9.columns = ['Violations', 'Number of Compounds']
else:
    table_3_9 = pd.DataFrame({'Violations': [0, 1, 2, 3, 4], 'Number of Compounds': [0, 0, 0, 0, 0]})

tables_dict['Table_3_9_Lipinski_Violations'] = table_3_9

# --- Figure 3.9: Filter Performance Chart ---
perf_data = pd.DataFrame({
    'Filter': ['Lipinski', 'Veber', 'Ghose'],
    'Pass (%)': [table_3_6.loc[0, 'Percentage (%)'], table_3_7.loc[0, 'Percentage (%)'], table_3_8.loc[0, 'Percentage (%)']],
    'Fail (%)': [table_3_6.loc[1, 'Percentage (%)'], table_3_7.loc[1, 'Percentage (%)'], table_3_8.loc[1, 'Percentage (%)']]
}).set_index('Filter')

perf_data.plot(kind='bar', figsize=(8, 5), color=['#2ecc71', '#e74c3c'], rot=0)
plt.title('Figure 3.9: Drug-Likeness Filter Performance Comparison', fontweight='bold', pad=12)
plt.ylabel('Percentage (%)')
plt.ylim(0, 100)
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('Figure_3_9_Filter_Performance.png', dpi=300)
plt.close()

print("✅ Tables 3.6–3.9 and Figure 3.9 generated.")

✅ Tables 3.6–3.9 and Figure 3.9 generated.


In [10]:
# ==============================================================================
# Step 8: QED Profiling and Candidate Prioritization
# ==============================================================================

qed_bins = [-1, 0.35, 0.67, 1.0]
qed_labels = ['Low (<0.35)', 'Moderate (0.35-0.67)', 'High (≥0.67)']
df['QED_Category'] = pd.cut(df['QED_Score'], bins=qed_bins, labels=qed_labels)

# --- Table 3.10: QED Categories ---
table_3_10 = df['QED_Category'].value_counts().reset_index()
table_3_10.columns = ['Category', 'Number']
table_3_10['Percentage (%)'] = ((table_3_10['Number'] / len(df)) * 100).round(2)
tables_dict['Table_3_10_QED_Categories'] = table_3_10

# --- Table 3.11: Top 20 Candidates ---
plant_col = 'Plant_Name' if 'Plant_Name' in df.columns else df.columns[0]
table_3_11 = df.nlargest(20, 'QED_Score')[['Compound_Name', plant_col, 'QED_Score']].reset_index(drop=True)
table_3_11.columns = ['Compound', 'Plant', 'QED']
table_3_11['QED'] = table_3_11['QED'].round(3)
tables_dict['Table_3_11_Top20_QED'] = table_3_11

# --- Figure 3.10: QED Histogram ---
plt.figure(figsize=(7, 4.5))
sns.histplot(df['QED_Score'], kde=True, color='#8e44ad', bins=20)
plt.axvline(0.67, color='r', linestyle='--', label='High Drug-Likeness Threshold (0.67)')
plt.title('Figure 3.10: Distribution of QED Scores', fontweight='bold')
plt.xlabel('QED Score')
plt.ylabel('Frequency')
plt.legend()
plt.savefig('Figure_3_10_QED_Distribution.png', dpi=300)
plt.close()

print("✅ Tables 3.10, 3.11, and Figure 3.10 generated.")

✅ Tables 3.10, 3.11, and Figure 3.10 generated.


In [11]:
# ==============================================================================
# Step 9: FDA Benchmarking and Principal Component Analysis
# ==============================================================================

# --- Table 3.12: FDA Reference Drugs ---
fda_data = [
    {"Compound": "Metformin", "Drug Class": "Biguanide", "MW": 129.16, "LogP": -1.43, "TPSA": 87.8, "HBD": 4, "HBA": 5, "RB": 2},
    {"Compound": "Glibenclamide", "Drug Class": "Sulfonylurea", "MW": 494.01, "LogP": 3.75, "TPSA": 112.0, "HBD": 2, "HBA": 5, "RB": 5},
    {"Compound": "Sitagliptin", "Drug Class": "DPP-4 Inhibitor", "MW": 407.31, "LogP": 0.60, "TPSA": 77.0, "HBD": 1, "HBA": 7, "RB": 5},
    {"Compound": "Dapagliflozin", "Drug Class": "SGLT2 Inhibitor", "MW": 408.87, "LogP": 2.34, "TPSA": 109.0, "HBD": 4, "HBA": 6, "RB": 5}
]
fda_df = pd.DataFrame(fda_data)
table_3_12 = fda_df[['Compound', 'Drug Class']]
tables_dict['Table_3_12_FDA_Benchmark'] = table_3_12

# --- Table 3.13: Comparison Summary ---
table_3_13 = pd.DataFrame({
    'Descriptor': ['MW', 'LogP', 'TPSA', 'HBD', 'HBA', 'RB'],
    'AfroDiabDB Mean': [df['Molecular_Weight'].mean(), df['LogP'].mean(), df['TPSA'].mean(), df['HBD'].mean(), df['HBA'].mean(), df['Rotatable_Bonds'].mean()],
    'FDA Drugs Mean': [fda_df['MW'].mean(), fda_df['LogP'].mean(), fda_df['TPSA'].mean(), fda_df['HBD'].mean(), fda_df['HBA'].mean(), fda_df['RB'].mean()]
}).round(2)
tables_dict['Table_3_13_FDA_Comparison'] = table_3_13

# --- Figure 3.11: Radar Plot ---
categories = ['MW', 'LogP', 'TPSA', 'HBD', 'HBA', 'RB']
fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
angles += angles[:1]

v1 = (table_3_13['AfroDiabDB Mean'] / table_3_13['AfroDiabDB Mean'].max()).tolist() + [(table_3_13['AfroDiabDB Mean'] / table_3_13['AfroDiabDB Mean'].max()).tolist()[0]]
v2 = (table_3_13['FDA Drugs Mean'] / table_3_13['AfroDiabDB Mean'].max()).tolist() + [(table_3_13['FDA Drugs Mean'] / table_3_13['AfroDiabDB Mean'].max()).tolist()[0]]

ax.plot(angles, v1, color='#2b5c8f', linewidth=2, label='AfroDiabDB')
ax.fill(angles, v1, color='#2b5c8f', alpha=0.25)
ax.plot(angles, v2, color='#e74c3c', linewidth=2, label='FDA Benchmark')
ax.fill(angles, v2, color='#e74c3c', alpha=0.25)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories)
plt.title('Figure 3.11: Radar Plot (AfroDiabDB vs. FDA Drugs)', fontweight='bold', pad=15)
plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))
plt.savefig('Figure_3_11_Radar_Plot.png', dpi=300)
plt.close()

# --- PCA Execution ---
pca_cols = ['Molecular_Weight', 'LogP', 'TPSA', 'HBA', 'HBD', 'Rotatable_Bonds']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[pca_cols].dropna())
pca = PCA(n_components=3)
pca_coords = pca.fit_transform(X_scaled)

df['PC1'] = pca_coords[:, 0]
df['PC2'] = pca_coords[:, 1]

# --- Table 3.14: PCA Variance Explained ---
table_3_14 = pd.DataFrame({
    'Component': ['PC1', 'PC2', 'PC3'],
    'Variance Explained (%)': (pca.explained_variance_ratio_ * 100).round(2)
})
tables_dict['Table_3_14_PCA_Variance'] = table_3_14

# --- Table 3.15: PCA Loadings Matrix ---
table_3_15 = pd.DataFrame(pca.components_[:2], columns=pca_cols, index=['PC1', 'PC2']).T.round(3).reset_index()
table_3_15.columns = ['Descriptor', 'PC1', 'PC2']
tables_dict['Table_3_15_PCA_Loadings'] = table_3_15

# --- Figure 3.12: Scree Plot ---
plt.figure(figsize=(6, 4))
plt.plot(['PC1', 'PC2', 'PC3'], pca.explained_variance_ratio_ * 100, marker='o', color='#27ae60')
plt.title('Figure 3.12: PCA Scree Plot', fontweight='bold')
plt.xlabel('Principal Component')
plt.ylabel('Variance Explained (%)')
plt.savefig('Figure_3_12_Scree_Plot.png', dpi=300)
plt.close()

# --- Figure 3.13: PCA Chemical Space Plot ---
plt.figure(figsize=(9, 6))
plt.scatter(df['PC1'], df['PC2'], c='#2b5c8f', alpha=0.7, s=60, label='AfroDiabDB Compounds')
plt.title('Figure 3.13: PCA Chemical Space Scatter Plot', fontweight='bold', pad=12)
plt.xlabel(f'PC1 ({table_3_14.loc[0, "Variance Explained (%)"]}%)')
plt.ylabel(f'PC2 ({table_3_14.loc[1, "Variance Explained (%)"]}%)')
plt.legend()
plt.tight_layout()
plt.savefig('Figure_3_13_PCA_Scatter_Plot.png', dpi=300)
plt.close()

print("✅ Tables 3.12–3.15 and Figures 3.11–3.13 generated.")

✅ Tables 3.12–3.15 and Figures 3.11–3.13 generated.


In [12]:

# ==============================================================================
# Step 10: FAIR Compliance, Excel Master Export & Drive Backup
# ==============================================================================

# --- Table 3.16: Data Availability & Repositories ---
table_3_16 = pd.DataFrame({
    'Resource': ['GitHub Repository', 'Zenodo Archive', 'Excel Database', 'CSV Database', 'SDF Dataset'],
    'Link / Format': [
        'https://github.com/LydiaFadele/AfroDiabDB',
        'https://zenodo.org/record/AfroDiabDB_v1.3',
        'AfroDiabDB_v1.3_final.xlsx',
        'AfroDiabDB_v1.3_final.csv',
        'AfroDiabDB_v1.3_structures.sdf'
    ]
})
tables_dict['Table_3_16_Database_Availability'] = table_3_16

# --- Figure 3.14: Data Distribution & Architecture ---
dot_arch = graphviz.Digraph('arch', format='png')
dot_arch.attr(rankdir='TB', size='6,6', dpi='300')
dot_arch.attr('node', shape='ellipse', style='filled', fillcolor='#EAFAF1', color='#27AE60', fontname='Helvetica')
dot_arch.node("A", "Literature Mining")
dot_arch.node("B", "AfroDiabDB Curation")
dot_arch.node("C", "GitHub Repository")
dot_arch.node("D", "Zenodo Archive")
dot_arch.node("E", "Researchers / End Users")
dot_arch.edges([("A", "B"), ("B", "C"), ("B", "D"), ("C", "E"), ("D", "E")])
dot_arch.render('Figure_3_14_Data_Architecture', cleanup=True)

# --- Consolidated Master Export (All 16 Tables) ---
excel_out = "AfroDiabDB_v1.3_Chapter3_Tables.xlsx"
with pd.ExcelWriter(excel_out, engine='openpyxl') as writer:
    for sheet, table_df in tables_dict.items():
        table_df.to_excel(writer, sheet_name=sheet[:31], index=False)

print(f"🎉 MASTER EXPORT COMPLETE: All 16 tables written to '{excel_out}'!")

# --- Sync to Google Drive ---
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
drive_folder = '/content/drive/MyDrive/AfroDiabDB_Chapter3_Final_Outputs'
os.makedirs(drive_folder, exist_ok=True)

all_outputs = glob.glob("*.png") + glob.glob("*.xlsx")
for file_path in all_outputs:
    shutil.copy(file_path, os.path.join(drive_folder, file_path))

print(f"🚀 SUCCESS! All 16 Tables and 14 Figures saved to Google Drive at: '{drive_folder}'")

🎉 MASTER EXPORT COMPLETE: All 16 tables written to 'AfroDiabDB_v1.3_Chapter3_Tables.xlsx'!
Mounted at /content/drive
🚀 SUCCESS! All 16 Tables and 14 Figures saved to Google Drive at: '/content/drive/MyDrive/AfroDiabDB_Chapter3_Final_Outputs'
